# 1장 — 문자에서 BPE까지 토크나이저 만들기

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/zerokaraLLM/blob/main/notebooks/ch01_tokenizer.ipynb)

이 노트북은 『밑바닥부터 시작하는 딥러닝 6』의 공식 코드 저장소를 기준으로 구성했습니다. T4에서 실행하기 어렵다는 이유로 알고리즘이나 모델 구조를 토이 버전으로 바꾸지 않습니다.

- 기준 upstream commit: `c9b6e2ed531b08dd9f451a091a34e9645148e2e2`
- 포함한 장 코드 파일 수: **9개**
- 함께 펼쳐서 보여주는 공통 모듈 수: **1개**


## 노트북 구성 원칙

1. 공식 `.py`의 모델 구조와 계산 로직을 그대로 유지합니다.
2. 함수·클래스·실행부를 셀 단위로 나눠 위에서 아래로 읽기 쉽게 배치합니다.
3. 일본어 자연어 주석은 한국어로 바꾸며, 변수명·수식·텐서 shape 같은 기술 표기는 유지합니다.
4. 공통 `codebot` / `storybot` 모듈도 외부 파일 뒤에 숨기지 않고 이 노트북에서 직접 확인할 수 있게 합니다.
5. T4에서 시간이 오래 걸리는 전체 학습 스케줄도 기본값 자체를 임의 축소하지 않습니다.


## 0. Colab 환경 준비

먼저 공식 저장소를 고정된 커밋으로 준비하고 현재 런타임의 GPU를 확인합니다.


In [ ]:
from pathlib import Path
import os
import subprocess

UPSTREAM_COMMIT = 'c9b6e2ed531b08dd9f451a091a34e9645148e2e2'
WORKDIR = Path('/content/deep-learning-from-scratch-6')

if not WORKDIR.exists():
    subprocess.run(['git', 'clone', '--quiet', 'https://github.com/oreilly-japan/deep-learning-from-scratch-6.git', str(WORKDIR)], check=True)
    subprocess.run(['git', '-C', str(WORKDIR), 'checkout', '--quiet', UPSTREAM_COMMIT], check=True)

os.chdir(WORKDIR)
print('작업 경로:', Path.cwd())

try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA 사용 가능:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('PyTorch 확인 중 오류:', exc)


## 1. 이 장에서 사용하는 공통 구현

장 코드가 import하는 로컬 모듈을 먼저 읽습니다. 긴 파일도 클래스·함수 단위로 나눠 표시합니다.


### `codebot/tokenizer.py`

이 파일은 이 장에서 사용하는 공통 구현입니다. 파일 전체를 숨기지 않고 구성 요소별로 나누어 확인합니다.


#### 필요한 라이브러리와 모듈 불러오기


In [ ]:
%%writefile codebot/tokenizer.py
import regex as re
from collections import defaultdict
import pickle
from tqdm import tqdm


#### `pretokenize()` 함수 구현


In [ ]:
%%writefile -a codebot/tokenizer.py


def pretokenize(text):
    pattern = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    return re.findall(pattern, text)


#### `count_pairs()` 함수 구현


In [ ]:
%%writefile -a codebot/tokenizer.py

def count_pairs(ids, counts=None):
    if counts is None:
        counts = defaultdict(int)

    for pair in zip(ids, ids[1:]):
        counts[pair] += 1
    return counts


#### `merge()` 함수 구현


In [ ]:
%%writefile -a codebot/tokenizer.py

def merge(ids, pair, new_id):
    merged_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            merged_ids.append(new_id)
            i += 2
        else:
            merged_ids.append(ids[i])
            i += 1
    return merged_ids


#### `train_bpe()` 함수 구현


In [ ]:
%%writefile -a codebot/tokenizer.py

def train_bpe(input_text, vocab_size, end_token="<|endoftext|>"):
    texts = input_text.split(end_token)

    ids_list = []
    for text in texts:
        for pretoken in pretokenize(text):
            ids_list.append(list(pretoken.encode("utf-8")))

    num_merges = vocab_size - 256 - 1
    merge_rules = {}

    for step in tqdm(range(num_merges), desc="Training BPE"):
        counts = defaultdict(int)
        for ids in ids_list:
            counts = count_pairs(ids, counts)

        if not counts:
            break

        # 참고: best_pair = max(counts, key=counts.get)
        best_pair = max(counts, key=lambda pair: (counts[pair], pair[0], pair[1]))

        new_id = 256 + step
        merge_rules[best_pair] = new_id

        for i in range(len(ids_list)):
            ids_list[i] = merge(ids_list[i], best_pair, new_id)

    return merge_rules


#### `BPETokenizer` 클래스 구현


In [ ]:
%%writefile -a codebot/tokenizer.py


class BPETokenizer:
    def __init__(self, merge_rules, end_token="<|endoftext|>"):
        self.merge_rules = merge_rules
        self.end_token = end_token
        self.end_token_id = 256 + len(merge_rules)

        self.id_to_bytes = {i: bytes([i]) for i in range(256)}
        for (id1, id2), new_id in merge_rules.items():
            self.id_to_bytes[new_id] = self.id_to_bytes[id1] + self.id_to_bytes[id2]
        self.id_to_bytes[self.end_token_id] = self.end_token.encode("utf-8")

        self.vocab_size = len(self.id_to_bytes)

    @staticmethod
    def load_from(filepath):
        with open(filepath, "rb") as f:
            merge_rules = pickle.load(f)
        return BPETokenizer(merge_rules)

    def _encode_text(self, text):
        ids = list(text.encode("utf-8"))
        for merge_pair, new_id in self.merge_rules.items():
            ids = merge(ids, merge_pair, new_id)
        return ids

    def encode(self, input_text, show_progress=False):
        pattern = '(' + re.escape(self.end_token) + ')'
        texts = re.split(pattern, input_text)
        all_ids = []

        # 이 코드 단계의 동작을 확인하는 예시
        texts = tqdm(texts, desc="Encoding") if show_progress else texts

        for text in texts:
            if text == self.end_token:
                all_ids.append(self.end_token_id)
            else:
                # 이 코드 단계의 동작을 확인하는 예시
                for pretoken in pretokenize(text):
                    ids = self._encode_text(pretoken)
                    all_ids.extend(ids)

        return all_ids

    def decode(self, ids):
        byte_list = [self.id_to_bytes[i] for i in ids]
        text_bytes = b"".join(byte_list)
        text = text_bytes.decode("utf-8", errors="replace")
        return text


## 2. 장별 실습 코드

공식 저장소의 장 코드를 파일 순서대로 모두 다룹니다.


## `ch01/01_char_tokenizer.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 설정 및 값 준비: `text`


In [ ]:
text = "hello世界😁"


### 실행 및 결과 확인


In [ ]:
print(list(text))  # 출력 예시


### 실행 및 결과 확인


In [ ]:

print(ord('h'))  # 참고: 104


### 실행 및 결과 확인


In [ ]:
print(ord('😁'))  # 참고: 128513


### 실행 및 결과 확인


In [ ]:

print(chr(104))    # 출력 예시: 'h'


### 실행 및 결과 확인


In [ ]:
print(chr(128513)) # 출력 예시: '😁'


### 설정 및 값 준비: `ids`


In [ ]:

ids = [ord(char) for char in list(text)]


### 실행 및 결과 확인


In [ ]:
print(ids)  # 출력 예시: [104, 101, 108, 108, 111, 19990, 30028, 128513]


### `CharTokenizer` 클래스 구현


In [ ]:

class CharTokenizer:
    def encode(self, text):
        return [ord(char) for char in text]

    def decode(self, ids):
        return ''.join([chr(i) for i in ids])


### 설정 및 값 준비: `tokenizer`


In [ ]:

tokenizer = CharTokenizer()


### 설정 및 값 준비: `text`


In [ ]:
text = "hello世界😁"


### 설정 및 값 준비: `ids`


In [ ]:

# 인코딩
ids = tokenizer.encode(text)


### 실행 및 결과 확인


In [ ]:
print(ids)  # 출력 예시: [104, 101, 108, 108, 111, 19990, 30028, 128513]


### 설정 및 값 준비: `decoded`


In [ ]:

# 디코딩
decoded = tokenizer.decode(ids)


### 실행 및 결과 확인


In [ ]:
print(decoded)  # 이 코드 단계의 동작을 확인하는 예시


## `ch01/02_byte_tokenizer.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 설정 및 값 준비: `encoded`


In [ ]:
# 출력 예시: 'A' 인 경우
encoded = 'A'.encode("utf-8")


### 실행 및 결과 확인


In [ ]:
print(encoded)        # 출력 예시: b'A'


### 실행 및 결과 확인


In [ ]:
print(list(encoded))  # 출력 예시: [65]


### 설정 및 값 준비: `encoded`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
encoded = 'あ'.encode("utf-8")


### 실행 및 결과 확인


In [ ]:
print(encoded)        # 출력 예시: b'\xe3\x81\x82'


### 실행 및 결과 확인


In [ ]:
print(list(encoded))  # 출력 예시: [227, 129, 130]


### 설정 및 값 준비: `ids`


In [ ]:

ids = [65]


### 설정 및 값 준비: `decoded`


In [ ]:
decoded = bytes(ids).decode("utf-8")


### 실행 및 결과 확인


In [ ]:
print(decoded)   # 출력 예시: 'A'


### `ByteTokenizer` 클래스 구현


In [ ]:


class ByteTokenizer:
    def encode(self, text):
        return list(text.encode("utf-8"))

    def decode(self, ids):
        return bytes(ids).decode("utf-8")


### 설정 및 값 준비: `tokenizer`


In [ ]:


# 사용 예시
tokenizer = ByteTokenizer()


### 설정 및 값 준비: `text`


In [ ]:
text = "hello世界😁"


### 설정 및 값 준비: `ids`


In [ ]:

# 인코딩
ids = tokenizer.encode(text)


### 실행 및 결과 확인


In [ ]:
print(ids)  # 출력 예시: [104, 101, 108, 108, 111, 228, 184, 150, 231, 149, 140, 240, 159, 152, 129]


### 설정 및 값 준비: `decoded`


In [ ]:

# 디코딩
decoded = tokenizer.decode(ids)


### 실행 및 결과 확인


In [ ]:
print(decoded)  # 이 코드 단계의 동작을 확인하는 예시


## `ch01/03_bpe_train.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
from collections import defaultdict


### `count_pairs()` 함수 구현


In [ ]:

def count_pairs(ids):
    counts = defaultdict(int)
    for pair in zip(ids, ids[1:]):
        counts[pair] += 1
    return counts


### 설정 및 값 준비: `ids`


In [ ]:

# 사용 예시
ids = [1, 2, 3, 1, 2]


### 설정 및 값 준비: `counts`


In [ ]:
counts = count_pairs(ids)


### 실행 및 결과 확인


In [ ]:
print(counts)  # 출력 예시: {(1, 2): 2, (2, 3): 1, (3, 1): 1}


### `merge()` 함수 구현


In [ ]:

def merge(ids, pair, new_id):
    merged_ids = []
    i = 0

    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            merged_ids.append(new_id)
            i += 2
        else:
            merged_ids.append(ids[i])
            i += 1

    return merged_ids


### 설정 및 값 준비: `ids`


In [ ]:

# 사용 예시
ids = [1, 2, 3, 1, 2]


### 설정 및 값 준비: `merged`


In [ ]:
merged = merge(ids, (1, 2), 4)


### 실행 및 결과 확인


In [ ]:
print(merged)  # 출력 예시: [4, 3, 4]


### `train_bpe()` 함수 구현


In [ ]:

def train_bpe(text, vocab_size):
    # 이 코드 단계의 동작을 확인하는 예시
    ids = list(text.encode("utf-8"))

    # 이 코드 단계의 동작을 확인하는 예시
    num_merges = vocab_size - 256  # 이 코드 단계의 동작을 확인하는 예시
    merge_rules = {}

    for step in range(num_merges):
        # 이 코드 단계의 동작을 확인하는 예시
        counts = count_pairs(ids)

        # 이 코드 단계의 동작을 확인하는 예시
        if not counts:
            break

        # 이 코드 단계의 동작을 확인하는 예시
        best_pair = max(counts, key=counts.get)
        # 참고: best_pair = max(counts, key=lambda pair: (counts[pair], pair[0], pair[1]))

        # 이 코드 단계의 동작을 확인하는 예시
        new_id = 256 + step
        merge_rules[best_pair] = new_id

        # 이 코드 단계의 동작을 확인하는 예시
        ids = merge(ids, best_pair, new_id)

    return merge_rules


### 설정 및 값 준비: `text`


In [ ]:

# 사용 예시
text = "Hello world! This is BPE training."


### 설정 및 값 준비: `merge_rules`


In [ ]:
merge_rules = train_bpe(text, vocab_size=260)


### 실행 및 결과 확인


In [ ]:
print(merge_rules)  # 출력 예시: {(105, 115): 256, (256, 32): 257, (105, 110): 258, (72, 101): 259}


## `ch01/04_bpe_tokenizer.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
from collections import defaultdict


### `count_pairs()` 함수 구현


In [ ]:

def count_pairs(ids):
    counts = defaultdict(int)
    for pair in zip(ids, ids[1:]):
        counts[pair] += 1
    return counts


### `merge()` 함수 구현


In [ ]:

def merge(ids, pair, new_id):
    merged_ids = []
    i = 0

    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            merged_ids.append(new_id)
            i += 2
        else:
            merged_ids.append(ids[i])
            i += 1

    return merged_ids


### `BPETokenizer` 클래스 구현


In [ ]:


class BPETokenizer:
    def __init__(self, merge_rules):
        self.merge_rules = merge_rules

        # 이 코드 단계의 동작을 확인하는 예시
        self.id_to_bytes = {i: bytes([i]) for i in range(256)}

        # 이 코드 단계의 동작을 확인하는 예시
        for (id1, id2), new_id in merge_rules.items():
            self.id_to_bytes[new_id] = self.id_to_bytes[id1] + self.id_to_bytes[id2]

        # 이 코드 단계의 동작을 확인하는 예시
        self.vocab_size = len(self.id_to_bytes)

    def encode(self, text):
        ids = list(text.encode("utf-8"))

        # 이 코드 단계의 동작을 확인하는 예시
        for merge_pair, new_id in self.merge_rules.items():
            ids = merge(ids, merge_pair, new_id)

        return ids

    def decode(self, ids):
        # 이 코드 단계의 동작을 확인하는 예시
        byte_list = [self.id_to_bytes[i] for i in ids]

        # 이 코드 단계의 동작을 확인하는 예시
        combined_bytes = b"".join(byte_list)

        # 이 코드 단계의 동작을 확인하는 예시
        text = combined_bytes.decode("utf-8", errors="replace")
        return text


### 설정 및 값 준비: `merge_rules`


In [ ]:

# 학습된의병합규칙
merge_rules = {(105, 115): 256, (256, 32): 257, (105, 110): 258, (72, 101): 259}


### 설정 및 값 준비: `tokenizer`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
tokenizer = BPETokenizer(merge_rules)


### 설정 및 값 준비: `text`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
text = "Hello世界😁"


### 설정 및 값 준비: `ids`


In [ ]:
ids = tokenizer.encode(text)


### 설정 및 값 준비: `decoded`


In [ ]:
decoded = tokenizer.decode(ids)


### 실행 및 결과 확인


In [ ]:

print(ids)  # 출력 예시: [259, 108, 108, 111, 228, 184, 150, 231, 149, 140, 240, 159, 152, 129]


### 실행 및 결과 확인


In [ ]:
print(decoded)  # 이 코드 단계의 동작을 확인하는 예시


## `ch01/05_special_token.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
from collections import defaultdict
import re


### `count_pairs()` 함수 구현


In [ ]:

# 참고: def count_pairs(ids):
# 참고: counts = defaultdict(int)
# 참고: for pair in zip(ids, ids[1:]):
# 참고: counts[pair] += 1
# 참고: return counts

def count_pairs(ids, counts=None):
    if counts is None:
        counts = defaultdict(int)

    for pair in zip(ids, ids[1:]):
        counts[pair] += 1
    return counts


### `merge()` 함수 구현


In [ ]:

def merge(ids, pair, new_id):
    merged_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            merged_ids.append(new_id)
            i += 2
        else:
            merged_ids.append(ids[i])
            i += 1
    return merged_ids


### `train_bpe()` 함수 구현


In [ ]:

def train_bpe(input_text, vocab_size, end_token="<|endoftext|>"):
    # 이 코드 단계의 동작을 확인하는 예시
    texts = input_text.split(end_token)
    ids_list = [list(text.encode("utf-8")) for text in texts]

    # 이 코드 단계의 동작을 확인하는 예시
    num_merges = vocab_size - 256 - 1
    merge_rules = {}

    for step in range(num_merges):
        # 이 코드 단계의 동작을 확인하는 예시
        counts = defaultdict(int)
        for ids in ids_list:
            counts = count_pairs(ids, counts)

        # 이 코드 단계의 동작을 확인하는 예시
        if not counts:
            break

        # 이 코드 단계의 동작을 확인하는 예시
        best_pair = max(counts, key=counts.get)
        # 참고: best_pair = max(counts, key=lambda pair: (counts[pair], pair[0], pair[1]))
        new_id = 256 + step
        merge_rules[best_pair] = new_id

        # 이 코드 단계의 동작을 확인하는 예시
        for i in range(len(ids_list)):
            ids_list[i] = merge(ids_list[i], best_pair, new_id)

    return merge_rules


### 설정 및 값 준비: `sample_text`


In [ ]:

# 사용 예시
sample_text = "Hello world!<|endoftext|>This is BPE training."


### 설정 및 값 준비: `merge_rules`


In [ ]:

merge_rules = train_bpe(sample_text, vocab_size=260)


### 실행 및 결과 확인


In [ ]:
print(merge_rules)  # 출력 예시: {(105, 115): 256, (256, 32): 257, (105, 110): 258}


### `BPETokenizer` 클래스 구현


In [ ]:


class BPETokenizer:
    def __init__(self, merge_rules, end_token="<|endoftext|>"):
        self.merge_rules = merge_rules
        self.end_token = end_token
        self.end_token_id = 256 + len(merge_rules)

        self.id_to_bytes = {i: bytes([i]) for i in range(256)}
        for (id1, id2), new_id in merge_rules.items():
            self.id_to_bytes[new_id] = self.id_to_bytes[id1] + self.id_to_bytes[id2]
        self.id_to_bytes[self.end_token_id] = self.end_token.encode("utf-8")

        self.vocab_size = len(self.id_to_bytes)

    def _encode_text(self, text):
        ids = list(text.encode("utf-8"))
        for merge_pair, new_id in self.merge_rules.items():
            ids = merge(ids, merge_pair, new_id)
        return ids

    def encode(self, input_text):
        pattern = '(' + re.escape(self.end_token) + ')'
        texts = re.split(pattern, input_text)
        all_ids = []

        for text in texts:
            if text == self.end_token:
                all_ids.append(self.end_token_id)
            else:
                ids = self._encode_text(text)
                all_ids.extend(ids)

        return all_ids

    def decode(self, ids):
        byte_list = [self.id_to_bytes[i] for i in ids]
        text_bytes = b"".join(byte_list)
        text = text_bytes.decode("utf-8", errors="replace")
        return text


### 설정 및 값 준비: `tokenizer`


In [ ]:


tokenizer = BPETokenizer(merge_rules)


### 설정 및 값 준비: `text`


In [ ]:

text = "Hello world!<|endoftext|>"


### 설정 및 값 준비: `ids`


In [ ]:
ids = tokenizer.encode(text)


### 설정 및 값 준비: `decoded`


In [ ]:
decoded = tokenizer.decode(ids)


### 실행 및 결과 확인


In [ ]:

print(ids)


### 실행 및 결과 확인


In [ ]:
print(decoded)


## `ch01/06_pretokenize.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
from collections import defaultdict
import regex as re
from tqdm import tqdm


### `pretokenize()` 함수 구현


In [ ]:


def pretokenize(text):
    # 이 코드 단계의 동작을 확인하는 예시
    pattern = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    return re.findall(pattern, text)


### `count_pairs()` 함수 구현


In [ ]:

def count_pairs(ids, counts=None):
    if counts is None:
        counts = defaultdict(int)

    for pair in zip(ids, ids[1:]):
        counts[pair] += 1
    return counts


### `merge()` 함수 구현


In [ ]:

def merge(ids, pair, new_id):
    merged_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            merged_ids.append(new_id)
            i += 2
        else:
            merged_ids.append(ids[i])
            i += 1
    return merged_ids


### `train_bpe()` 함수 구현


In [ ]:

def train_bpe(input_text, vocab_size, end_token="<|endoftext|>"):
    # 이 코드 단계의 동작을 확인하는 예시
    texts = input_text.split(end_token)

    # 이 코드 단계의 동작을 확인하는 예시
    ids_list = []
    for text in texts:
        for pretoken in pretokenize(text):  # 이 코드 단계의 동작을 확인하는 예시
            ids_list.append(list(pretoken.encode("utf-8")))  # 이 코드 단계의 동작을 확인하는 예시

    # 이 코드 단계의 동작을 확인하는 예시
    num_merges = vocab_size - 256 - 1
    merge_rules = {}

    for step in tqdm(range(num_merges), desc="Training BPE"):  # 이 코드 단계의 동작을 확인하는 예시
        counts = defaultdict(int)
        for ids in ids_list:
            counts = count_pairs(ids, counts)

        if not counts:
            break

        best_pair = max(counts, key=counts.get)
        # 참고: best_pair = max(counts, key=lambda pair: (counts[pair], pair[0], pair[1]))

        new_id = 256 + step
        merge_rules[best_pair] = new_id

        for i in range(len(ids_list)):
            ids_list[i] = merge(ids_list[i], best_pair, new_id)

    return merge_rules


### `BPETokenizer` 클래스 구현


In [ ]:


class BPETokenizer:
    def __init__(self, merge_rules, end_token="<|endoftext|>"):
        self.merge_rules = merge_rules
        self.end_token = end_token
        self.end_token_id = 256 + len(merge_rules)

        self.id_to_bytes = {i: bytes([i]) for i in range(256)}
        for (id1, id2), new_id in merge_rules.items():
            self.id_to_bytes[new_id] = self.id_to_bytes[id1] + self.id_to_bytes[id2]
        self.id_to_bytes[self.end_token_id] = self.end_token.encode("utf-8")

        self.vocab_size = len(self.id_to_bytes)

    def _encode_text(self, text):
        ids = list(text.encode("utf-8"))
        for merge_pair, new_id in self.merge_rules.items():
            ids = merge(ids, merge_pair, new_id)
        return ids

    def encode(self, input_text, show_progress=False):
        pattern = '(' + re.escape(self.end_token) + ')'
        texts = re.split(pattern, input_text)
        all_ids = []

        # 이 코드 단계의 동작을 확인하는 예시
        texts = tqdm(texts, desc="Encoding") if show_progress else texts

        for text in texts:
            if text == self.end_token:
                all_ids.append(self.end_token_id)
            else:
                # 이 코드 단계의 동작을 확인하는 예시
                for pretoken in pretokenize(text):
                    ids = self._encode_text(pretoken)
                    all_ids.extend(ids)

        return all_ids

    def decode(self, ids):
        byte_list = [self.id_to_bytes[i] for i in ids]
        text_bytes = b"".join(byte_list)
        text = text_bytes.decode("utf-8", errors="replace")
        return text


### 설정 및 값 준비: `sample_text`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
sample_text = "Say hello! Why hello? Just hello.<|endoftext|>Good morning!"


### 설정 및 값 준비: `merge_rules`


In [ ]:

merge_rules = train_bpe(sample_text, vocab_size=270)


### 설정 및 값 준비: `tokenizer`


In [ ]:
tokenizer = BPETokenizer(merge_rules)


### 설정 및 값 준비: `text`


In [ ]:

# 인코딩/디코딩
text = "Say hello!"


### 설정 및 값 준비: `ids`


In [ ]:
ids = tokenizer.encode(text)


### 설정 및 값 준비: `decoded`


In [ ]:
decoded = tokenizer.decode(ids)


### 실행 및 결과 확인


In [ ]:

print(ids)


### 실행 및 결과 확인


In [ ]:
print(decoded)


### 반복 실행


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
for token_id in ids:
    print(f"{token_id} -> '{tokenizer.decode([token_id])}'")


## `ch01/07_tiny_codes.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import os, sys


### 실행 및 결과 확인


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))


### 실행 및 결과 확인


In [ ]:
sys.path.append('.')


### 필요한 라이브러리와 모듈 불러오기


In [ ]:

import pickle
from codebot.tokenizer import train_bpe


### 설정 및 값 준비: `vocab_size`


In [ ]:

vocab_size = 1000  # 어휘 크기


### 설정 및 값 준비: `text`


In [ ]:
text = open("codebot/tiny_codes.txt").read()


### 설정 및 값 준비: `merge_rules`


In [ ]:
merge_rules = train_bpe(text, vocab_size)


### 실행 코드


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
with open("codebot/merge_rules.pkl", "wb") as f:
    pickle.dump(merge_rules, f)


## `ch01/08_eval.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import os, sys


### 실행 및 결과 확인


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))


### 실행 및 결과 확인


In [ ]:
sys.path.append('.')


### 필요한 라이브러리와 모듈 불러오기


In [ ]:

from codebot.tokenizer import BPETokenizer


### 설정 및 값 준비: `tokenizer`


In [ ]:


tokenizer = BPETokenizer.load_from("codebot/merge_rules.pkl")


### 실행 및 결과 확인


In [ ]:

print("最初に学習された10個:")


### 반복 실행


In [ ]:
for token_id in range(256, 266):
    byte_seq = tokenizer.id_to_bytes[token_id]
    text = byte_seq.decode("utf-8")
    print(f"  ID {token_id}: '{text}'")


### 실행 및 결과 확인


In [ ]:

print("\n最後に学習された10個:")


### 반복 실행


In [ ]:
for token_id in range(990, 1000):
    byte_seq = tokenizer.id_to_bytes[token_id]
    text = byte_seq.decode("utf-8")
    print(f"  ID {token_id}: '{text}'")


### 설정 및 값 준비: `sample_text`


In [ ]:


# 이 코드 단계의 동작을 확인하는 예시
sample_text = open("codebot/tiny_codes.txt").read()[:10000]  # 이 코드 단계의 동작을 확인하는 예시


### 설정 및 값 준비: `byte_count`


In [ ]:

byte_count = len(sample_text.encode("utf-8"))


### 설정 및 값 준비: `ids`


In [ ]:
ids = tokenizer.encode(sample_text)


### 설정 및 값 준비: `ids_count`


In [ ]:
ids_count = len(ids)


### 설정 및 값 준비: `compression_ratio`


In [ ]:
compression_ratio = byte_count / ids_count


### 실행 및 결과 확인


In [ ]:

print("\n=== 圧縮効率 ===")


### 실행 및 결과 확인


In [ ]:
print(f"バイト数: {byte_count:,}")


### 실행 및 결과 확인


In [ ]:
print(f"トークン数: {ids_count:,}")


### 실행 및 결과 확인


In [ ]:
print(f"圧縮率: {compression_ratio:.2f}倍（平均 {compression_ratio:.2f} バイト/トークン）")


### 실행 및 결과 확인


In [ ]:


# 이 코드 단계의 동작을 확인하는 예시
"""
import tiktoken

text = open("codebot/tiny_codes.txt").read()[:10000]
byte_count = len(text.encode("utf-8"))

for name, encoding_name in [('GPT-2', 'gpt2'), ('cl100k_base', 'cl100k_base')]:
    encoding = tiktoken.get_encoding(encoding_name)
    token_count = len(encoding.encode(text, allowed_special={'<|endoftext|>'}))
    ratio = byte_count / token_count
    print(f"{name}: 語彙サイズ {encoding.n_vocab:,}, 圧縮率 {ratio:.2f}倍")
"""


## `ch01/09_bpe_encode.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import os, sys


### 실행 및 결과 확인


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))


### 실행 및 결과 확인


In [ ]:
sys.path.append('.')


### 필요한 라이브러리와 모듈 불러오기


In [ ]:

import numpy as np
from codebot.tokenizer import BPETokenizer


### 설정 및 값 준비: `tokenizer`


In [ ]:


# 이 코드 단계의 동작을 확인하는 예시
tokenizer = BPETokenizer.load_from("codebot/merge_rules.pkl")


### 설정 및 값 준비: `text`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
text = open("codebot/tiny_codes.txt").read()


### 설정 및 값 준비: `ids`


In [ ]:
ids = tokenizer.encode(text, show_progress=True)


### 설정 및 값 준비: `ids_array`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
ids_array = np.array(ids, dtype=np.uint16)


### 실행 및 결과 확인


In [ ]:
ids_array.tofile("codebot/tiny_codes.bin")


### 실행 및 결과 확인


In [ ]:

print(f"トークンID数: {len(ids_array)}")


### 실행 및 결과 확인


In [ ]:
print(f"最初の20個のトークンID: {ids_array[:20]}")


## T4 실행 메모

위 코드는 공식 구현의 모델 구조·알고리즘·기본 하이퍼파라미터를 보존합니다. 학습 시간이 긴 셀은 T4에서도 실행 자체는 가능할 수 있지만 전체 스텝 완주에는 시간이 많이 필요할 수 있습니다. 이 노트북은 빠른 실행을 위해 모델을 임의로 축소하거나 핵심 계산을 생략하지 않습니다.
